# Config

In [1]:
!git config --global --add safe.directory /tmp/Repository/VRID_language_proyect

In [ ]:
import os

# Ruta a la que quieres mover el path
nueva_ruta = "/tmp/Repository/VRID_language_proyect/BERT"

# Cambiar el directorio actual
os.chdir(nueva_ruta)

# Confirmar que cambió
print("Directorio actual:", os.getcwd())

Directorio actual: /tmp/Repository/VRID_language_proyect/BERT


# 1) Preprocesamiento de los datos


In [3]:
import pandas as pd
from preprocess.preprocess import clean_text, check_deleted_expressions


# 1) Cargar datos
path = "/tmp/final_project/datasets/"
path_output = "/tmp/final_project/output/interdiciplinario"
filePATH = os.path.join(path, "all_data.csv")
df = pd.read_csv(filePATH)

# 2) Guardar qué secuencias de palabras del resumen serán eliminadas al aplicar get_expressions_to_delete()
list_texts = df["Resumen"].to_list()
df_deleted = check_deleted_expressions(list_texts)
savepath=os.path.join(path_output, "deleted_re.xlsx")
df_deleted.to_excel(savepath, index=False)

# 3) Eliminar elementos duplicados
df = df.drop_duplicates("Código VRID")

# 4) Preprocesar los datos
#Columnas que se van a preprocesar
#Nombre fila seleccionada:Columna que se creará para guardar resultado
cols = {
    "Título": "Titulo_trad",
    "Resumen": "Resumen_trad",
    "Keywords": "keywords_trad",
    "Facultad del Proyecto": "Facultad_del_Proyecto_trad",
    "Depto Persona": "Depto_Persona_trad",
}
#Preprocesamiento de datos
df[list(cols.values())] = df[list(cols.keys())].applymap(clean_text)

# 2) Traducción del texto

In [4]:
from preprocess.translate import translator, final_clean, detect_language
from transformers import MarianMTModel, MarianTokenizer
import time

#Crear columna de registro de idioma: 
# True: Texto en español, False: Texto en inglés
df["Español"]=detect_language(df["Resumen_trad"])

#1. Cargar modelo de traducción
model_name = "Helsinki-NLP/opus-mt-es-en"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)
trans = translator(model, tokenizer)

#Columnas que se van a traducir
#Nombre fila seleccionada:Columna que se creará para guardar resultado
cols = {
    "Titulo_trad": "Titulo_trad",
    "Resumen_trad": "Resumen_trad",
    "keywords_trad": "keywords_trad",
    "Facultad_del_Proyecto_trad": "Facultad_del_Proyecto_trad",
    "Depto_Persona_trad": "Depto_Persona_trad",
}

#2. Traducción de columnas
#####Estoy trabajando en mejorar esta parte para que sea más rápida con paralelización por batches
start = time.time()
for src, dst in cols.items():
    df[dst] = trans.translate_parallel(df[src].to_list(), batch_size=8)
end = time.time()

print(f"Tiempo total de traducción: {end - start:.2f} segundos")

/usr/local/lib/python3.10/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
/usr/local/lib/python3.10/dist-packages/torch/_utils.py:830: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


Usando dispositivo: cuda


Traduciendo: 100%|██████████| 119/119 [00:26<00:00,  4.44batch/s]


Tiempo total de traducción: 826.09 segundos


In [5]:
# Guardado de resultados
savepath=os.path.join(path, "data_translated.csv")
df.to_csv(savepath, index=False, encoding="utf-8-sig")
